# Quickstart for strategy owners
Copy this notebook into your strategy's folder. Everything below runs against the real QQQ data in `data/raw/QQQ.csv`.

**Never evaluate on `test` while developing.**

In [ ]:
import pandas as pd
from qqq.data import load_prices, split
from qqq import features as F
from qqq.evaluate import evaluate, grid_search, check_no_lookahead, backtest_strategy
pd.set_option('display.float_format', '{:.4f}'.format)

prices = load_prices()
train, val, _ = split(prices)
prices.tail()

## 1. Write / import your strategy
Start from `src/qqq/strategies/_template.py`, or iterate here and move it into `src/` when it works.

In [ ]:
from qqq.strategies._template import MyStrategy   # <- replace with your class
strat = MyStrategy(window=200)

## 2. Evaluate on train + validation (vs buy-and-hold)

In [ ]:
evaluate(strat)[['annualized_mean_return','annualized_return','annualized_volatility','sharpe_ratio','max_drawdown','cumulative_return','exposure','n_position_changes']]

## 3. Parameter search: rank on TRAIN, confirm the top few on VALIDATION

In [ ]:
gs = grid_search(MyStrategy, {'window': [20, 50, 100, 150, 200]}, period='train')
gs.head(10)

In [ ]:
grid_search(MyStrategy, {'window': list(gs.window.head(3))}, period='validation')

## 4. Look-ahead check (also runs in CI once registered)

In [ ]:
check_no_lookahead(strat); print('no look-ahead')

## 5. Inspect positions & equity

In [ ]:
res = backtest_strategy(strat, prices)
ax = res.equity.loc[:'2021'].plot(logy=True, figsize=(10,4), label=strat.name)
ax2 = ax.twinx(); res.positions.loc[:'2021'].plot(ax=ax2, alpha=0.3, color='grey')
ax.legend();